In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql.functions import col, current_timestamp, lit, row_number
from etl.transformations.common import (
    create_spark,
    list_batches,
    read_parquet,
    write_clickhouse,
)

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_crop(
    crops_df: DataFrame,
    categories_df: DataFrame,
) -> DataFrame:
    """
    Build dim_crop rows from changed crops and categories.
    """

    return crops_df.join(
        categories_df,
        crops_df.category_id == categories_df.id,
        "inner",
    ).select(
        crops_df.id.alias("crop_id"),
        crops_df.name,
        crops_df.description,
        crops_df.category_id.alias("crop_category_id"),
        categories_df.name.alias("category_name"),
        lit(0).alias("is_high_value"),
        current_timestamp().alias("_loaded_at"),
    )

In [3]:

def read_current_snapshot(
    spark: SparkSession,
    bucket: str,
    table_name: str,
    primary_key: str = "id",
    version_column: str = "updated_at",
) -> DataFrame:
    """
    Reconstruct the current table state from all incremental batches.

    Airflow stores only changed rows in each batch, so the latest batch is
    not necessarily a complete snapshot. This helper reads every available
    batch and keeps only the newest version of each primary key.

    Example:

        batch1
        id=1
        id=2

        batch2
        id=2 (updated)

        result
        id=1
        id=2 (updated)
    """

    base_path = f"s3a://{bucket}/raw/postgres/{table_name}/"

    batches = list_batches(
        spark,
        base_path,
    )

    paths = [f"{base_path}{batch}" for batch in batches]

    df = read_parquet(
        spark,
        *paths,
    )

    window = Window.partitionBy(primary_key).orderBy(col(version_column).desc())

    return (
        df.withColumn(
            "_row_number",
            row_number().over(window),
        )
        .filter(
            col("_row_number") == 1,
        )
        .drop("_row_number")
    )

In [4]:
def main():

    spark = create_spark("load_dim_crop")

    try:
        crops_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "crops",
        )

        categories_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "crop_categories",
        )

        crops_df.show()
        categories_df.show()

        dim_crop_df = transform_dim_crop(
            crops_df,
            categories_df,
        )

        write_clickhouse(
            dim_crop_df,
            "dim_crop",
        )

    finally:
        spark.stop()

In [5]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/21 23:56:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/21 23:56:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---+-----------+------------------+--------------------+----------+----------+
| id|category_id|              name|         description|created_at|updated_at|
+---+-----------+------------------+--------------------+----------+----------+
|  1|          2|            Salata|Aromatic herb use...|1784672803|1784675324|
|  2|          2|              Mint|Refreshing herb u...|1784672803|1784672803|
|  3|          2|           Parsley|Versatile herb fo...|1784672803|1784672803|
|  4|          2|          Cilantro|Herb with citrusy...|1784672803|1784672803|
|  5|          2|             Thyme|Small-leaved herb...|1784672803|1784672803|
|  6|          2|          Rosemary|Fragrant, woody h...|1784672803|1784672803|
|  7|          1|   Romaine Lettuce|Crisp, elongated ...|1784672803|1784672803|
|  8|          1|Butterhead Lettuce|Soft, tender leav...|1784672803|1784672803|
|  9|          1|   Arugula Lettuce|Peppery, tangy gr...|1784672803|1784672803|
| 10|          1|           Spinach|Tend

26/07/21 23:56:43 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/22 00:17:31 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-b57c61c0-1811-4d48-aabd-7295d6584580/pyspark-73c4f3c7-83f3-41ec-a79f-6ae445e7e59f. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-b57c61c0-1811-4d48-aabd-7295d6584580/pyspark-73c4f3c7-83f3-41ec-a79f-6ae445e7e59f
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManager$.$ano